In [ ]:
import os
import wikipediaapi
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [ ]:
from dotenv import load_dotenv

# To securely load my OpenAI api key from .env file
load_dotenv()


## Preparing Data for Knowledge Base

In [ ]:
wiki = wikipediaapi.Wikipedia(
    user_agent='MasterThesisResearch',
    language='en'
)

# Sections to exclude
EXCLUDE = ["References", "External links", "Further reading", "See also", "Notes", "Sources"]


def process_sections(sections, level=2):
    md_content = ""
    for s in sections:

        if s.title in EXCLUDE:
             continue
        
        md_content += f"{'#' * level} {s.title}\n\n"
        md_content += f"{s.text}\n\n"

        if s.sections:
            md_content += process_sections(s.sections, level + 1)

    return md_content


games = [
    "Agricola (board game)", 
    "Arkham Horror", 
    "Azul (board game)", 
    "Carcassonne (board game)", 
    "Codenames (board game)",
    "Catan", 
    "Dune (board game)",    
    "Everdell", 
    "Gloomhaven", 
    "Monopoly (game)",      
    "Pandemic (board game)", 
    "Pictionary", 
    "Ra (board game)",      
    "Root (board game)",   
    "Scythe (board game)", 
    "Spirit Island (board game)", 
    "Splendor (board game)", 
    "Terraforming Mars (board game)", 
    "Ticket to Ride (board game)", 
    "Wingspan (board game)"
]


if not os.path.exists("data_md"):
        os.makedirs("data_md")

for game_title in games:

    page = wiki.page(game_title)

    if page.exists():

        final_md = f"# {page.title}\n\n"

        final_md += f"{page.summary}\n\n"

        final_md += process_sections(page.sections)
    

        filename = game_title.lower().replace(" (board game)", "").replace(" (game)", "").replace(" ", "_") + ".md"

        with open(f"data_md/{filename}", "w", encoding="utf-8") as f:
            f.write(final_md)

            print(f"Successfully saved: {filename}!")

    else:
        print(f"The page {page} does not exist.")

### Document Chunking

In [ ]:
data_path = "data_md"

headers_to_split_on = [
    ("#", "Game"),
    ("##", "Section"),
    ("###", "Subsection"),
    ("####", "Detail"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

all_chunks = []

for filename in os.listdir(data_path):
    if filename.endswith(".md"):
        file_path = os.path.join(data_path, filename)

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

            chunks = md_splitter.split_text(content)

            # Keeping track of the original file source
            for chunk in chunks:
                chunk.metadata["source"] = filename

            all_chunks.extend(chunks)




### Creating Embeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


embeddings = OpenAIEmbeddings(model="text-embedding-3-small")



### Creating Vector Store

In [ ]:
persist_directory = "new_vector_store"

vector_db = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

## Creating Golden Dataset

In [ ]:
import os
import pandas as pd
from deepeval.synthesizer import Synthesizer
from deepeval.models.llms import GPTModel
from deepeval.synthesizer.config import StylingConfig
import random


# Customizing prompt
easy_style = StylingConfig(
    task="""Generate a factual or procedural question based on the context provided. 
    STRICT RULE: Both the question and the 'Gold Answer' must be extracted EXCLUSIVELY from the provided text. 
    If a fact is not explicitly written in the context, you ARE FORBIDDEN from including it in the answer. 
    Do not use external knowledge and do not ask "what if" questions. 
    IMPORTANT: You MUST mention the game name (found after 'SOURCE:') in the question."""
)

model = GPTModel(model="gpt-4o-mini", temperature=0)
synthesizer = Synthesizer(model=model, styling_config=easy_style, cost_tracking=True)


# Testing on a single document to see if it works properly
target_file = "catan.md"
document_contexts = [
    chunk.page_content 
    for chunk in all_chunks 
    if target_file in chunk.metadata.get('source', '')
]


# Choosing 3 random chunks
num_samples = 3
if len(document_contexts) > num_samples:
    random_chunks = random.sample(document_contexts, num_samples)
else:
    random_chunks = document_contexts


# Generating questions and gold answers from chosen chunks
try:
    generated_goldens = synthesizer.generate_goldens_from_contexts(
        contexts=[[c] for c in random_chunks], 
        max_goldens_per_context=1,
        
    )

    synthetic_data = []
    for golden in generated_goldens:
        synthetic_data.append({
            "question": golden.input,
            "answer": golden.expected_output,
            "context": golden.context[0] 
        })

    results_df = pd.DataFrame(synthetic_data)
    print(results_df[['question', 'context']].head())

except Exception as e:
    print(f"DeepEval Error: {e}")


 


In [ ]:
for i, row in results_df.iterrows():
    print(f"Question {i+1}: {row['question']}")
    print(f"Answer {i+1}: {row['answer']}")
    print("-" * 50)

In [ ]:
# Choosing random chunks 
selected_chunks = random.sample(all_chunks, 150)


sample_contexts = []
for chunk in selected_chunks:
    game = chunk.metadata.get('source')
    context = f"SOURCE: {game.replace('.md', '')} | CONTENT: {chunk.page_content}"
    sample_contexts.append([context])

In [ ]:
# Generating synthetic evaluation dataset
try: 
    generated_goldens = synthesizer.generate_goldens_from_contexts(
        contexts=sample_contexts,
        max_goldens_per_context=1
    )

    final_data = []
    for g in generated_goldens:
        raw_context = g.context[0]

        parts = raw_context.split(" | CONTENT: ", 1)
        source = parts[0].replace("SOURCE: ", "").strip().split('/')[-1]
        clean_text = parts[1]

        final_data.append({
            "source": source,
            "question": g.input,
            "answer": g.expected_output,
            "context": clean_text
        })


    df_final = pd.DataFrame(final_data)
    df_final.to_csv("gold_ds.csv", index=False)

except Exception as e:
    print(f"Greska: {e}")

In [ ]:
# Loading the same data set after manual removal of bad examples
df_all = pd.read_csv('gold_ds_final.csv')

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Load database from dir
vector_db = Chroma(persist_directory="./new_vector_store", embedding_function=embeddings)

## Creating RAG Chain and Framework for Evaluation

In [ ]:
# Creating retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3}) 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# System prompt template for the LLM within the RAG system
template = """You are a helpful AI assistant. 
Your answer must be based strictly on the retrieved context provided below. 
If the context does not contain the necessary information to answer the question, 
clearly state that the information does not exist in the provided documents. 
Do not use any outside knowledge or assumptions.

Context:
{context}

Question: {input}

Answer:"""

custom_prompt = ChatPromptTemplate.from_template(template)

In [ ]:
import csv
import json
import os
import time
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, ContextualPrecisionMetric
from langchain_classic.evaluation import load_evaluator


CSV_FILE = "eval_rag_all.csv"
JSONL_CONTEXT_FILE = "retrieval_contexts.jsonl"


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # For testing gpt 4o I just changed model here and appropriate file names
eval_model = GPTModel(model="gpt-4o")
eval_llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Defining deepeval metrics
relevancy = AnswerRelevancyMetric(threshold=0.7, model=eval_model, include_reason=False)
precision = ContextualPrecisionMetric(threshold=0.7, model=eval_model, include_reason=False)

# Defining langchain groundedness
langchain_evaluator = load_evaluator(
    "labeled_score_string",
    llm=eval_llm,
    criteria={
        "grounded": "The answer should only use facts supported by the provided context.",
        "honesty": "If context is insufficient, the answer should say it doesn't know."
    },
)

# Creating RAG chain
combine_docs_chain = create_stuff_documents_chain(llm, custom_prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

df_all = pd.read_csv("gold_ds_final.csv")
total_questions = len(df_all)

In [ ]:
# Checkpoint mechanism to resume evaluation if the API crashes or disconnects
def get_last_id(filename):
    if not os.path.exists(filename):
        return -1
    try:
        df_check = pd.read_csv(filename)
        if df_check.empty:
            return -1
        return df_check['ID'].max()
    except:
        return -1


if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'ID', 'QUESTION', 'RAG_OUTPUT', 'GOLD_STANDARD', 
            'RELEVANCY', 'PRECISION', 'GROUNDED_SCORE'
        ])

In [ ]:
last_id = get_last_id(CSV_FILE)

print(f"Starting evaluation of all {total_questions} questions.")
if last_id >= 0:
    print(f"Resuming from index: {last_id + 1}")


with open(CSV_FILE, 'a', newline='', encoding='utf-8', buffering=1) as csv_f, \
     open(JSONL_CONTEXT_FILE, 'a', encoding='utf-8', buffering=1) as jsonl_f:
    
    csv_writer = csv.writer(csv_f)

    for index, row in df_all.iterrows():
        # Skipping examples that are already processed and saved
        if index <= last_id:
            continue

        query = row['question']
        reference_answer = row['answer']
        gold_context = row['context']

        try:
            print(f"\n--- Working on: {index + 1}/{total_questions} ---")
            print(f"Question: {query}")
            
            # Invoking chain and getting the answer
            response = rag_chain.invoke({"input": query})
            generated_answer = response["answer"]
            
            
            retrieval_context_list = [doc.page_content for doc in response["context"]]
            retrieval_context_text = "\n\n".join(retrieval_context_list)
            
            # Creating deepeval test case
            test_case = LLMTestCase(
                input=query,
                actual_output=generated_answer,
                retrieval_context=retrieval_context_list,
                expected_output=reference_answer
            )
            
            # Calculating deepeval metrics
            print("Calculating DeepEval metrics...")
            relevancy.measure(test_case)
            precision.measure(test_case)
            
            # Calculating langchain groundedness
            print("Calculating LangChain Groundedness...")
            lc_result = langchain_evaluator.evaluate_strings(
                prediction=generated_answer,
                input=query,
                reference=retrieval_context_text
            )
            
            raw_score = lc_result.get("score")  
            
            # Normalizing groundedness to scale [0,1]
            if raw_score is not None:
                final_grounded_score = (raw_score - 1) / 9
            else:
                final_grounded_score = 0.0

            
            csv_writer.writerow([
                index,
                query,
                generated_answer.replace('\n', ' '),
                reference_answer.replace('\n', ' '),
                round(relevancy.score, 2),
                round(precision.score, 2),
                round(final_grounded_score, 2),
            ])
            
            # Writing data into jsonl for later calculating hit rate and mrr
            context_data = {
                "id": index,
                "question": query,
                "gold_context": gold_context, 
                "retrieved_contexts": retrieval_context_list 
            }
            jsonl_f.write(json.dumps(context_data, ensure_ascii=False) + '\n')
            
            print(f"ID {index} successfully saved.")
            
            
            

        except Exception as e:
            print(f"\nError at index {index}: {e}")
            raise e

### Calculating HitRate

In [ ]:
import json

JSONL_CONTEXT_FILE = "retrieval_contexts.jsonl"

with open(JSONL_CONTEXT_FILE, 'r', encoding='utf-8') as f:
    lines = f.readlines()

total_cases = len(lines)
hits_at_3 = 0

for line in lines:
    data = json.loads(line)
    golden = data['gold_context'].strip()
    retrieved_list_top3 = data['retrieved_contexts']
    
    # Check if golden chunk is inside 3 retrieved chunks
    is_hit = 0
    for retrieved_chunk in retrieved_list_top3:
        retrieved_chunk = retrieved_chunk.strip()
        if golden in retrieved_chunk or retrieved_chunk in golden:
            is_hit = 1
            hits_at_3 += 1
            break  

    print(is_hit)

hit_rate_3 = hits_at_3 / total_cases

print(f"Hit Rate @ 3: {hit_rate_3:.4f} ({hit_rate_3 * 100:.2f}%)")

In [ ]:
hits_at_1 = 0

for line in lines:
    data = json.loads(line)
    golden = data['gold_context'].strip()
    retrieved_chunk = data['retrieved_contexts'][0]
    
    is_hit = 0
    retrieved_chunk = retrieved_chunk.strip()
    if golden in retrieved_chunk or retrieved_chunk in golden:
        is_hit = 1
        hits_at_1 += 1
           

    print(is_hit)

hit_rate_1 = hits_at_1 / total_cases

print(f"Hit Rate @ 1: {hit_rate_1:.4f} ({hit_rate_1 * 100:.2f}%)")

### Calculating MRR

In [ ]:
total_rr = 0

for line in lines:
    data = json.loads(line)
    golden = data['gold_context'].strip()
    retrieved_list = data['retrieved_contexts']
    
    found_rank = None
    for index, retrieved_chunk in enumerate(retrieved_list):
        retrieved_chunk = retrieved_chunk.strip()
        if golden in retrieved_chunk or retrieved_chunk in golden:
            found_rank = index + 1
            break
            
    # Calculating RR
    if found_rank is not None:
        rr = 1.0 / found_rank
    else:
        rr = 0.0
        
    total_rr += rr
    
    print(f"{rr:.2f}")


mrr = total_rr / total_cases

print(f"MRR @ 3: {mrr:.4f} ({mrr * 100:.2f}%)")


### RERANKER

In [ ]:
import csv
import json
import os
import time
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from deepeval.test_case import LLMTestCase
from langchain_classic.evaluation import load_evaluator
from deepeval.metrics import AnswerRelevancyMetric, ContextualPrecisionMetric

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever

In [ ]:
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from deepeval.models.llms import GPTModel


In [ ]:
CSV_FILE = "eval_rag_reranker.csv"
JSONL_CONTEXT_FILE = "retrieval_contexts_reranker.jsonl"

base_retriever = retriever
base_retriever.search_kwargs = {"k": 6}

# Loading BGE Reranker model
gold_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-large")

compressor = CrossEncoderReranker(model=gold_model, top_n=3)

advanced_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # For testing gpt 4o I just changed model here and appropriate file names
eval_model = GPTModel(model="gpt-4o")
eval_llm = ChatOpenAI(model="gpt-4o", temperature=0)

relevancy = AnswerRelevancyMetric(threshold=0.7, model=eval_model, include_reason=False)
precision = ContextualPrecisionMetric(threshold=0.7, model=eval_model, include_reason=False)


combine_docs_chain = create_stuff_documents_chain(llm, custom_prompt)
rag_chain_reranker = create_retrieval_chain(advanced_retriever, combine_docs_chain)

df_all = pd.read_csv("gold_ds_final.csv")
total_questions = len(df_all)

In [ ]:
def get_last_id(filename):
    if not os.path.exists(filename):
        return -1
    try:
        df_check = pd.read_csv(filename)
        if df_check.empty:
            return -1
        return df_check['ID'].max()
    except:
        return -1

if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'ID', 'QUESTION', 'RAG_OUTPUT', 'GOLD_STANDARD', 
            'RELEVANCY', 'PRECISION', 'GROUNDED_SCORE'
        ])

last_id = get_last_id(CSV_FILE)

print(f"Starting evaluation with reranker on {total_questions} questions.")
if last_id >= 0:
    print(f"Resuming evaluation at index: {last_id + 1}")


with open(CSV_FILE, 'a', newline='', encoding='utf-8', buffering=1) as csv_f, \
     open(JSONL_CONTEXT_FILE, 'a', encoding='utf-8', buffering=1) as jsonl_f:
    
    csv_writer = csv.writer(csv_f)

    for index, row in df_all.iterrows():
        # Skipping examples that are already processed and saved
        if index <= last_id:
            continue

        query = row['question']
        reference_answer = row['answer']
        gold_context = row['context']

        try:
            print(f"\n--- Working on: {index + 1}/{total_questions} ---")
            print(f"Question: {query}")
            
            # Invoking chain and getting the answer
            response = rag_chain_reranker.invoke({"input": query})
            generated_answer = response["answer"]
            
            
            retrieval_context_list = [doc.page_content for doc in response["context"]]
            retrieval_context_text = "\n\n".join(retrieval_context_list)
            
            # Creating deepeval test case
            test_case = LLMTestCase(
                input=query,
                actual_output=generated_answer,
                retrieval_context=retrieval_context_list,
                expected_output=reference_answer
            )
            
            # Calculating deepeval metrics
            print("Calculating DeepEval metrics...")
            relevancy.measure(test_case)
            precision.measure(test_case)
            
            # Calculating langchain groundedness
            print("Calculating LangChain Groundedness...")
            lc_result = langchain_evaluator.evaluate_strings(
                prediction=generated_answer,
                input=query,
                reference=retrieval_context_text
            )
            
            raw_score = lc_result.get("score")
            # Normalizing groundedness score to scale [0,1]
            final_grounded_score = (raw_score - 1) / 9 if raw_score is not None else 0.0

            
            csv_writer.writerow([
                index,
                query,
                generated_answer.replace('\n', ' '),
                reference_answer.replace('\n', ' '),
                round(relevancy.score, 2),
                round(precision.score, 2),
                round(final_grounded_score, 2),
            ])
            
            # Writing data into jsonl for later calculating hit rate and mrr
            context_data = {
                "id": index,
                "question": query,
                "gold_context": gold_context, 
                "retrieved_contexts": retrieval_context_list 
            }
            jsonl_f.write(json.dumps(context_data, ensure_ascii=False) + '\n')
            
            print(f"ID {index} successfully saved.")
            
            
            time.sleep(2)

        except Exception as e:
            print(f"\n[STOP] Error at index {index}: {e}")
            raise e

In [ ]:
import json

JSONL_CONTEXT_FILE = "retrieval_contexts_reranker.jsonl"

with open(JSONL_CONTEXT_FILE, 'r', encoding='utf-8') as f:
    lines = f.readlines()

total_cases = len(lines)
hits_at_3 = 0

for line in lines:
    data = json.loads(line)
    golden = data['gold_context'].strip()
    retrieved_list_top3 = data['retrieved_contexts']
    
    # Check if golden chunk is inside 3 retrieved chunks
    is_hit = 0
    for retrieved_chunk in retrieved_list_top3:
        retrieved_chunk = retrieved_chunk.strip()
        if golden in retrieved_chunk or retrieved_chunk in golden:
            is_hit = 1
            hits_at_3 += 1
            break  

    print(is_hit)

hit_rate_3 = hits_at_3 / total_cases

print(f"Hit Rate @ 3: {hit_rate_3:.4f} ({hit_rate_3 * 100:.2f}%)")

In [ ]:
hits_at_1 = 0

for line in lines:
    data = json.loads(line)
    golden = data['gold_context'].strip()
    retrieved_chunk = data['retrieved_contexts'][0]
    
    
    is_hit = 0
    retrieved_chunk = retrieved_chunk.strip()
    if golden in retrieved_chunk or retrieved_chunk in golden:
        is_hit = 1
        hits_at_1 += 1
           

    print(is_hit)

hit_rate_1 = hits_at_1 / total_cases

print(f"Hit Rate @ 1: {hit_rate_1:.4f} ({hit_rate_1 * 100:.2f}%)")

In [ ]:
total_rr = 0

for line in lines:
    data = json.loads(line)
    golden = data['gold_context'].strip()
    retrieved_list = data['retrieved_contexts']
    
    found_rank = None
    
    for index, retrieved_chunk in enumerate(retrieved_list):
        retrieved_chunk = retrieved_chunk.strip()
        if golden in retrieved_chunk or retrieved_chunk in golden:
            found_rank = index + 1
            break
            
    
    if found_rank is not None:
        rr = 1.0 / found_rank
    else:
        rr = 0.0
        
    total_rr += rr
    
    
    print(f"{rr:.2f}")


mrr = total_rr / total_cases

print(f"MRR @ 3: {mrr:.4f} ({mrr * 100:.2f}%)")
